# PRACTICE 2: PREDICTING PRODUCT SALES
## PHẦN 3: HUẤN LUYỆN MÔ HÌNH, ĐÁNH GIÁ & GIẢI THÍCH (MODELING, EVALUATION & EXPLAINABILITY)

> **Mục tiêu kỹ thuật:**
> - Huấn luyện 3 mô hình hồi quy From Scratch tự viết bằng NumPy thuần túy: Linear Regression (OLS + L2), Decision Tree Regressor và Random Forest Regressor (Ensemble Bagging).
> - Kiểm định chéo 5-Fold Cross Validation trên tập Train để đo lường tính ổn định mô hình.
> - Đánh giá độc lập trên 20.571 mẫu Test set, nghịch đảo expm1 về thang đo INR kèm cơ chế chống tràn số.
> - Bóc tách Top 15 đặc trưng chi phối doanh số và đóng gói mô hình Quán quân vào `models/saved_models/`.

---
## Khai báo Thư viện & Nạp Dữ liệu Đã Xử lý từ Notebook 2
**Mục tiêu:** Nạp gói nhị phân `train_test_data.joblib` và dây chuyền tiền xử lý `preprocessor.joblib`.

In [ ]:
# Khai báo thư viện & Nạp gói dữ liệu
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent if Path.cwd().name in ['scripts', 'notebooks'] else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from scripts.feature_engineering import load_pipeline_and_splits
from scripts.model_training import create_scratch_models, build_training_pipelines, evaluate_cv_models, tune_grid_search, get_default_param_grids
from scripts.model_evaluation import evaluate_holdout_test, extract_feature_importances, export_evaluation_artifacts, plot_actual_vs_predicted, plot_residuals, plot_performance_comparison, plot_feature_importances

splits_path = BASE_DIR / 'data' / 'processed' / 'train_test_data.joblib'
prep_path = BASE_DIR / 'models' / 'saved_models' / 'preprocessor.joblib'

# load_pipeline_and_splits nhận (data_path, preprocessor_path)
data_bundle = load_pipeline_and_splits(splits_path, prep_path)
X_train = data_bundle['X_train']
X_test = data_bundle['X_test']
y_train = data_bundle['y_train']
y_test = data_bundle['y_test']
y_train_raw = data_bundle['y_train_raw']
y_test_raw = data_bundle['y_test_raw']
preprocessor = data_bundle['preprocessor']

print(f"✓ Nạp thành công bundle: Train {X_train.shape} | Test {X_test.shape}")

**Đúc kết nạp dữ liệu:**
* **Sẵn sàng 100%:** Toàn bộ 82.281 mẫu Train và 20.571 mẫu Test cùng đối tượng ColumnTransformer đã sẵn sàng huấn luyện.

---
## Mục 6: Huấn luyện Mô hình & Tối ưu Siêu tham số

### 6.1 Khởi tạo 3 Mô hình Scratch & Đánh giá Chéo 5-Fold Cross Validation
**Mục tiêu:** Khởi tạo 3 mô hình tự viết bằng NumPy thuần túy và kiểm định chéo 5-Fold Cross Validation trên tập Train để đo lường độ ổn định.

In [ ]:
models = create_scratch_models()
pipelines = build_training_pipelines(preprocessor, models=models)

print("Đang chạy kiểm định chéo 5-Fold Cross Validation...")
cv_summary_df = evaluate_cv_models(pipelines, X_train, y_train, cv=5, sample_size=10000)
display(cv_summary_df)

**Đúc kết kiểm định chéo:**
* **Độ ổn định phương sai rất cao:** Cả 3 mô hình đều đạt độ lệch chuẩn rất thấp (sigma < 0.01), chứng minh không có hiện tượng học vẹt (overfitting).
* **Random Forest dẫn đầu CV:** CV R2 đạt 0.5484 với độ lệch chuẩn nhỏ nhất (± 0.0092), thể hiện sự vững vàng vượt trội của mô hình Ensemble Bagging.

### 6.2 Tinh chỉnh Siêu tham số (GridSearchCV 3-Fold trên Mẫu đại diện 10.000 đơn hàng) & Lưu trữ Mô hình Tối ưu
**Mục tiêu:** Quét tự động lưới siêu tham số trên mẫu đại diện 10.000 dòng để tìm nhanh cấu hình tối ưu (trong ~1.5 phút), sau đó áp dụng tham số tối ưu huấn luyện toàn diện trên toàn bộ 82.281 mẫu Train và lưu vào `models/saved_models/`.

In [ ]:
# Tinh chỉnh siêu tham số thực thụ qua GridSearchCV và huấn luyện mô hình tối ưu
saved_models_dir = BASE_DIR / 'models' / 'saved_models'
param_grids = get_default_param_grids()

trained_models, tuning_summary = tune_grid_search(
    pipelines,
    param_grids=param_grids,
    X_train=X_train,
    y_train=y_train,
    cv=3,
    subsample_size=10000,
    random_state=42,
    models_dir=saved_models_dir,
    return_summary=True
)

display(tuning_summary)
print(f"✓ Đã huấn luyện và lưu trữ các mô hình tối ưu vào: {saved_models_dir}")

**Đúc kết tinh chỉnh siêu tham số:**
* **Quét siêu tham số tối ưu:** Thực nghiệm 3-Fold CV trên mẫu đại diện 10.000 mẫu giúp tìm nhanh cấu hình tối ưu cho Random Forest (`max_depth=8, n_estimators=20`) và Decision Tree (`max_depth=7, min_samples_split=10`) trong thời gian chưa đầy 1.5 phút.
* **Huấn luyện toàn diện vững chắc:** Mô hình tối ưu được cập nhật và huấn luyện hoàn chỉnh trên toàn bộ 82.281 đơn hàng, lưu trữ thành công `best_model.joblib` vào `models/saved_models/`.

---
## Mục 7: Đánh giá Mô hình trên Tập Kiểm tra (Model Evaluation) & Phân tích lỗi

### 7.1 Đánh giá Độc lập trên Tập Kiểm tra (20.571 mẫu Hold-out Test) kèm Nghịch đảo expm1
**Mục tiêu:** Dự đoán trên tập Test độc lập, chuyển đổi nghịch đảo về thang đo tiền tệ INR và đo lường R2, RMSE, MAE, MSE.

In [ ]:
test_metrics_df = evaluate_holdout_test(trained_models, X_test, y_test, y_test_raw)
print("--- KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP KIỂM THỬ ĐỘC LẬP (20.571 MẪU TEST) ---")
display(test_metrics_df)

**Đúc kết nghiệm thu tập kiểm thử:**
* **Random Forest Regressor đoạt Quán quân:** Đạt R2 (Thực tế) = 0.4911, R2 (Log) = 0.5640, MAE thấp nhất (139.21 INR) và RMSE thấp nhất (195.58 INR).
* **Decision Tree Regressor về Á quân:** Đạt R2 (Thực tế) = 0.4871, R2 (Log) = 0.5592, bám đuổi rất sát và có tốc độ suy luận nhanh nhất.

### 7.2 Phân tích Sai số Thực tế: Biểu đồ Thực tế vs Dự đoán, Phần dư & So sánh Hiệu năng
**Mục tiêu:** Trực quan hóa Actual vs Predicted, Residual Plot và biểu đồ cột đối sánh 3 mô hình để nhận diện Quán quân.

In [ ]:
images_dir = BASE_DIR / 'docs' / 'images'
images_dir.mkdir(parents=True, exist_ok=True)

# 1. Biểu đồ so sánh 3 mô hình
plot_performance_comparison(test_metrics_df, save_path=images_dir / 'model_03_performance_comparison.png')

# 2. Lấy dự đoán mô hình Quán quân Random Forest
rf_pipe = trained_models['Random Forest Regressor']
y_pred_log_rf = rf_pipe.predict(X_test)
y_pred_inr_rf = np.clip(np.expm1(np.clip(y_pred_log_rf, -10.0, 15.0)), 0.0, None)

# 3. Biểu đồ Actual vs Predicted & Residual Plot
plot_actual_vs_predicted(y_test_raw, y_pred_inr_rf, max_points=1000, save_path=images_dir / 'model_01_actual_vs_predicted.png')
plot_residuals(y_test_raw, y_pred_inr_rf, max_points=1000, save_path=images_dir / 'model_02_residuals.png')

**Đúc kết phân tích sai số:**
* **Hội tụ phân bố cân bằng:** Phần dư trung bình xấp xỉ 0 INR chứng minh mô hình không có thiên lệch hệ thống, bảo vệ an toàn cho ước tính dòng tiền.
* **Độ tin cậy cao ở dải phổ thông:** Mật độ dự báo khớp chính xác nhất ở vùng doanh số 300 – 1.200 INR (chiếm hơn 90% giao dịch toàn sàn).

### 7.3 Giải thích Mô hình: Top 15 Đặc trưng Chi phối Doanh số Mạnh nhất của Mô hình Quán quân
**Mục tiêu:** Trích xuất và trực quan hóa Top 15 đặc trưng quan trọng nhất từ mô hình Quán quân Random Forest.

In [ ]:
top_15_feats = extract_feature_importances(trained_models['Random Forest Regressor'], top_n=15)
plot_feature_importances(top_15_feats, top_n=15, save_path=images_dir / 'model_04_feature_importances.png')
display(top_15_feats)

**Đúc kết mức độ quan trọng đặc trưng:**
* **Ngành hàng định hình giá cơ sở:** Category_kurta và Category_Top là 2 thuộc tính chi phối mạnh nhất phân khúc giá sản phẩm.
* **Quy mô mua sắm & Mùa vụ:** Số lượng đặt mua (Qty) và yếu tố chu kỳ tháng/ngày đóng góp đáng kể vào tổng giá trị đơn hàng.

### 7.4 Đóng gói Mô hình Quán quân (best_model.joblib) & Siêu dữ liệu Nghiệm thu (metadata.json)
**Mục tiêu:** Xuất tệp metadata.json nghiệm thu toàn diện và kiểm tra tính toàn vẹn của mô hình best_model.joblib.

In [ ]:
rf_metrics = test_metrics_df[test_metrics_df['Mô hình'] == 'Random Forest Regressor'].iloc[0].to_dict()

meta_file = export_evaluation_artifacts(
    model_name='Random Forest Regressor (Ensemble Bagging Scratch)',
    metrics_dict=rf_metrics,
    top_features=top_15_feats,
    output_dir=saved_models_dir,
    models_comparison=test_metrics_df
)
print(f"✓ Đã nghiệm thu và xuất thành công siêu dữ liệu tại: {meta_file}")

**Đúc kết nghiệm thu và đóng gói:**
* **Đóng gói Quán quân:** Lưu trữ thành công `best_model.joblib` (Random Forest) kèm siêu dữ liệu kiểm định đầy đủ tại `metadata.json`.
* **Sẵn sàng triển khai:** Pipeline và mô hình tối ưu đã hoàn tất nghiệm thu kỹ thuật, sẵn sàng phục vụ suy luận tại `notebooks/demo.ipynb`.